In [ ]:
# cs781 phase 1, model 3 (post-ln version)
# decoder only, rope + gqa (10 q / 2 kv heads) + swiglu, weighted norm after each sublayer, 3 head mtp, tied embedding / output head
#
# only 2 changes from the pre-norm model 3, everything else (data, windows, mtp loss, optimiser, decoding) is the same:
#   1. the norm goes after the residual add: x = norm(x + F(x)) instead of x + F(norm(x))
#   2. rmsnorm -> WeightedNorm (layernorm with a learned weight and bias, computed in fp32)
#   that adds 12,160 params (64,728,960 -> 64,741,120) and uses a new MODEL_TAG / work dir,
#   so the pre-norm checkpoints are never resumed
#
# changes from model 2: new 50k byte level bpe (tokenizer.json), 8 layers, ffn 1536, dropout 0.1, 3 mtp heads
# (head k predicts token t+k, shared tied output matrix). the loss is computed in chunks with checkpointing because
# 4 x 50k logits for an 8x1024 micro batch won't fit in 16gb. decoding only uses head 1, so the decode v2 code
# (beam / mbr / bertscore mbr) works unchanged
#
# saved under WORK_DIR/checkpoints so a dead session can resume:
#   latest.pt / latest_meta.json   full state (model, optimizer, scaler, position)
#   best.pt / best_meta.json       best next token val loss so far
#   train_log.csv, progress.json   progress
#   decoding_sweep_v2.json         recipe picked on val, reused for the same checkpoint
#   test_gen_partial.json          test generations so far, a crashed decode run continues from here
# to resume: add the previous version's output as an input of the next run and run again,
# the newest latest.pt of this model is picked up (model 1/2 files are ignored because of the model tag)
#
# set MODE below: "train" (train + predict, resumes automatically) or "decode" (predict only, from best.pt)
import os
import csv
import sys
import json
import math
import time
import zlib
import shutil
import hashlib
import zipfile
import subprocess
from collections import Counter
from dataclasses import dataclass, asdict, field
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from huggingface_hub import hf_hub_download, list_repo_files, login

# config
MODE = "train"                     # train = train (resumes) then predict, decode = predict only
MODEL_TAG = "decoder_only_rope_gqa_swiglu_mtp3_postln_wnorm_v1"

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = "/kaggle/working/cs781_phase1_model3_postln" if IS_KAGGLE else "./cs781_phase1_model3_postln"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
CACHE_DIR = os.path.join(WORK_DIR, "cache")
INPUT_ROOT = "/kaggle/input"

# output files, same names and layout as the starter code
OUTPUT_DIR = "/kaggle/working/cs781_phase1" if IS_KAGGLE else "./cs781_phase1"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

# hf dataset, same repo and file paths as the starter code
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
TEST_FILE_ALT = "Phase_1_Pretraining/Test/Phase1- Test.json"
SOURCES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}
# share of training windows per source (same as model 2, not the char mix used for bpe training)
MIX = {"supreme_court": 0.35, "high_courts": 0.30, "tribunals": 0.20, "acts": 0.15}

# don't paste a token in a notebook you share, use the kaggle secret HF (or the HF_TOKEN env var)
HF_TOKEN_FALLBACK = ""

# tokenizer
VOCAB_SIZE = 50_000                # real tokenizer size (ids 0..49,999)
VOCAB_PAD_MULTIPLE = 64            # embedding rows are rounded up to a multiple of this (50,048), the extra ids are banned when decoding
ENCODE_BATCH = 64                  # docs tokenized per rust call when building the window cache
CHARS_PER_WINDOW_CAP = 9_000       # only the first cap*9000 chars of a doc can give windows, so don't tokenize more

# sequence / dataset
CONTEXT_LENGTH = 1024
STRIDE = 768
WINDOW = CONTEXT_LENGTH + 1        # 1024 inputs + 1 extra so the labels can be shifted by one
MAX_WINDOWS = 400_000              # training windows per epoch
VAL_WINDOWS = 2_000
MAX_WIN_PER_DOC = 8
VAL_MAX_WIN_PER_DOC = 2
VAL_MOD = 40                       # doc is val if crc32(source:idx) % VAL_MOD == 0
MIN_DOC_CHARS = 4000

# transformer
D_MODEL = 640
N_LAYERS = 8
N_HEADS = 10
N_KV_HEADS = 2
HEAD_DIM = 64
FFN_HIDDEN = 1536
DROPOUT = 0.1
assert D_MODEL == N_HEADS * HEAD_DIM and N_HEADS % N_KV_HEADS == 0

# multi token prediction
# mtp depth 4: head k (k = 1..4) predicts token t+k, head 1 is the normal next token head
# every head is z_k = WeightedNorm_k(x + W_k x) (residual projection 640 -> 640), then the shared tied embedding
MTP_HEADS = 3
MTP_WEIGHTS = [1.0, 0.5, 0.25]
assert len(MTP_WEIGHTS) == MTP_HEADS
MTP_POS_FRAC = 0.25                 # below 1.0 trains heads 2..4 on a random subset of positions (cheaper), 1.0 = all positions
CE_CHUNK = 2048                    # tokens per cross entropy chunk (keeps the fp32 logits small)
IGNORE = -100

# optimisation
EPOCHS = 4
MICRO_BS = 16                       # use 4 (and ACCUM 8) if you get cuda oom
ACCUM = 2
EFF_BS = MICRO_BS * ACCUM
LR = 3e-4
PEAK_LR = LR
MIN_LR_RATIO = 0.1
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.10
GRAD_CLIP = 1.0
BETAS = (0.9, 0.95)
SEED = 1234

LOG_EVERY = 50          # optimizer steps
EVAL_EVERY = 1000       # optimizer steps (val loss, maybe a new best)
SAVE_EVERY = 500        # optimizer steps (mid epoch "latest" checkpoint)
MAX_RUNTIME_H = 11.3    # stop (after saving) before kaggle's 12h limit

# generation / decoding
MAX_NEW_TOKENS = 256
PROMPT_TOKENS = CONTEXT_LENGTH - MAX_NEW_TOKENS     # prompt keeps its last 768 tokens
TARGET_WORDS = 128
RUN_PREDICTION = True      # in train mode, predict after training finishes
RUN_SELF_TEST = True       # cached decoding vs plain forward pass, runs once at start (takes seconds)

DECODE_MODE = "auto"        # auto = sweep on val windows and use the best, or greedy / beam / mbr / mbr_bs
QUICK_TEST = False          # True = quick smoke test (32 val windows, 16 test prompts, no zip)
BEAM_WIDTH = 4
NO_REPEAT_NGRAM_V2 = 4      # in bpe tokens (the 50k byte level vocab gives ~1.3 tokens per word)
MBR_SAMPLES = 12            # samples per prompt for mbr
MBR_TEMPERATURE = 0.8
MBR_TOP_P = 0.95
MBR_W_UNI = 0.7             # mbr score = 0.7 * unigram f1 + 0.3 * bigram f1
BS_MBR_SAMPLES = 24         # samples per prompt for the bertscore mbr
EMBED_MODEL = "roberta-large"   # model bert-score uses for lang="en", change it if the grader uses another one
EMBED_LAYER = 17            # layer bert-score uses in roberta-large
VAL_SWEEP_N = 128           # val windows used to compare recipes
ROW_BUDGET = 96             # how many sequences to decode at once, lower it (like 48) on cuda oom
EXTEND_TOKENS = 192         # extra tokens to generate when a continuation has under 128 words
EXTEND_ROUNDS = 3
PARTIAL_SAVE_SEC = 60       # how often (at most) the test generations get written to disk

AMP = {"enabled": False, "dtype": torch.float16}


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"


# finding files
def walk_find(filename, roots):
    # all dirs under roots that contain `filename`
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files:
                hits.append(d)
    return hits


def hf_token():
    tok = os.environ.get("HF_TOKEN")      # don't hard code a token, use the env var or the kaggle secret "HF"
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        return HF_TOKEN_FALLBACK or None


def hf_login():
    # hf login, token from env HF_TOKEN or the kaggle secret HF
    tok = hf_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        login(token=tok)
        log("Hugging Face login successful.")
    else:
        log("warning: no HF token found (set the Kaggle secret 'HF'); trying anonymous access")
    return tok


# tokenizer: the byte level bpe from the tokenizer notebook (hf tokenizers, tokenizer.json)
WORD_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+"""
)


class LegalBPE:
    def __init__(self, vocab_path, merges_path, meta_path=None):
        # load vocab
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)

        # load merges
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)

         # tokenizer fingerprint to validate the window cache
        h = hashlib.md5()

        for path in (vocab_path, merges_path, meta_path):
            if path and os.path.exists(path):
                with open(path, "rb") as f:
                    h.update(f.read())

        self.fingerprint = h.hexdigest()[:12]

        
        # merge rank, lower = merged earlier
        self.ranks = {
            (a, b): i
            for i, (a, b) in enumerate(merges)
        }

        # id -> token
        self.inv = {
            int(i): t
            for t, i in self.vocab.items()
        }

        self.vocab_size = max(self.inv) + 1

        assert len(self.inv) == self.vocab_size, \
            "vocab ids are not contiguous"

        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        self.bos_id = self.eos_id
        self.mask_id = self.eos_id

        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }
        # tokenizer metadata
        meta = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)

        for key, val in (
            ("pad_id", self.pad_id),
            ("unk_id", self.unk_id),
            ("eos_id", self.eos_id),
            ("bos_id", self.bos_id),
            ("mask_id", self.mask_id),
        ):
            if key in meta and meta[key] != val:
                raise ValueError(
                    f"meta.json {key}={meta[key]} disagrees with vocab ({val})"
            )

        if meta.get("vocab_size") not in (None, self.vocab_size):
            log(
                f"warning: meta.json vocab_size="
                f"{meta['vocab_size']} but vocab has "
                f"{self.vocab_size}"
            )

        # decoder only, so bos is just eos
        self.bos_id = self.eos_id
        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }

        self.is_end = [
            self.inv[i].endswith("</w>")
            for i in range(self.vocab_size)
        ]

        # cache bpe results for repeated words
        self.cache = {}

    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None:
            return hit

        # characters + the word end marker
        syms = list(word) + ["</w>"]

        # apply the merges
        while len(syms) > 1:
            best = None
            best_rank = None

            for i in range(len(syms) - 1):
                r = self.ranks.get(
                    (syms[i], syms[i + 1])
                )

                if r is not None and (
                    best_rank is None or r < best_rank
                ):
                    best = (
                        syms[i],
                        syms[i + 1]
                    )
                    best_rank = r

            if best is None:
                break

            merged = best[0] + best[1]

            out = []
            i = 0

            while i < len(syms):
                if (
                    i < len(syms) - 1
                    and syms[i] == best[0]
                    and syms[i + 1] == best[1]
                ):
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1

            syms = out

        ids = [
            self.vocab.get(s, self.unk_id)
            for s in syms
        ]

        # don't let the cache grow forever
        if len(self.cache) > 3_000_000:
            self.cache.clear()

        self.cache[word] = ids
        return ids

    def encode(self, text):
        ids = []

        for w in WORD_PATTERN.findall(text.lower()):
            ids.extend(self._word_ids(w))

        return ids

    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]

    def decode(self, ids):
        toks = [
            self.inv.get(int(i), "")
            for i in ids
            if int(i) not in self.special_ids
        ]

        return "".join(toks).replace("</w>", " ").strip()

def to_words(ids, tok, limit=TARGET_WORDS):
    text = tok.decode(ids).replace("\ufffd", "")      # half of a byte level token decodes to U+FFFD
    return " ".join(text.split()[:limit])


def n_words(ids, tok):
    return len(tok.decode(ids).replace("\ufffd", "").split())


def load_tokenizer():
    roots = [INPUT_ROOT, os.getcwd(), WORK_DIR]

    for d in walk_find("vocab_50k.json", roots):
        vocab_path = os.path.join(d, "vocab_50k.json")
        merges_path = os.path.join(d, "merges_50k.json")
        meta_path = os.path.join(d, "meta_50k.json")

        if os.path.exists(merges_path):
            log(f"tokenizer files found in {d}")

            tok = LegalBPE(
                vocab_path,
                merges_path,
                meta_path
            )

            log(
                f"vocab={tok.vocab_size} "
                f"pad={tok.pad_id} "
                f"unk={tok.unk_id} "
                f"eos={tok.eos_id}"
            )

            return tok

    raise FileNotFoundError(
        "vocab_50k.json + merges_50k.json "
        "not found under /kaggle/input"
    )

# streaming reader for the huge json files (never keeps a whole file in ram)
def local_chunks(path, size=1 << 24):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        while True:
            piece = f.read(size)
            if not piece:
                return
            yield piece


def open_source(name):
    # download the train file with hf_hub_download (like the starter code) and stream it from disk
    # (the starter's json.load keeps the whole file in ram, the streaming parser below doesn't)
    train_file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=SOURCES[name],
        repo_type="dataset",
        token=hf_token(),
    )
    log(f"[{name}] training file: {train_file_path}")
    return local_chunks(train_file_path)


def stream_records(chunk_iter):
    # yields every record of a top level json dict {id: rec} or list [rec, ...]
    dec = json.JSONDecoder()
    buf, pos, eof = "", 0, False

    def more():
        nonlocal buf, pos, eof
        try:
            piece = next(chunk_iter)
        except StopIteration:
            eof = True
            return
        buf = buf[pos:] + piece
        pos = 0

    def skip_ws():
        nonlocal pos
        while True:
            while pos < len(buf) and buf[pos] in " \r\n\t,":
                pos += 1
            if pos < len(buf) or eof:
                return
            more()

    def parse_value():
        nonlocal pos
        while True:
            try:
                obj, end = dec.raw_decode(buf, pos)
                pos = end
                return obj
            except json.JSONDecodeError:
                if eof:
                    raise
                more()

    skip_ws()
    if pos >= len(buf):
        return
    is_dict = buf[pos] == "{"
    pos += 1
    while True:
        skip_ws()
        if pos >= len(buf) or buf[pos] in "}]":
            return
        if is_dict:
            parse_value()          # the key
            skip_ws()
            if pos >= len(buf) or buf[pos] != ":":
                raise ValueError("malformed json: expected ':' after key")
            pos += 1
            skip_ws()
        yield parse_value()


def record_text(rec):
    if isinstance(rec, dict):
        t = rec.get("text")
        return t if isinstance(t, str) else None
    return rec if isinstance(rec, str) else None


# sliding windows
def cut_windows(ids, max_n):
    # overlapping WINDOW token windows, one every STRIDE tokens from the start of the doc
    out, start, n = [], 0, len(ids)
    while len(out) < max_n and start + WINDOW <= n:
        out.append(ids[start:start + WINDOW])
        start += STRIDE
    return out


def build_windows(tok):
    train_q = {k: int(MAX_WINDOWS * v) for k, v in MIX.items()}
    val_q = {k: max(1, int(VAL_WINDOWS * v)) for k, v in MIX.items()}
    train_out, val_out = [], []
    for name in SOURCES:
        t0 = time.time()
        st = {"tr": 0, "va": 0, "seen": 0, "skipped": 0}
        pending = []                                   # (is_val, text) waiting for the next batched tokenizer call

        def flush():
            if not pending:
                return
            encs = tok.encode_batch([t for _, t in pending])
            for (is_val, _), ids in zip(pending, encs):
                for w in cut_windows(ids, VAL_MAX_WIN_PER_DOC if is_val else MAX_WIN_PER_DOC):
                    arr = np.asarray(w, dtype=np.uint16)
                    if is_val and st["va"] < val_q[name]:
                        val_out.append(arr)
                        st["va"] += 1
                    elif not is_val and st["tr"] < train_q[name]:
                        train_out.append(arr)
                        st["tr"] += 1
            pending.clear()

        chunks = open_source(name)
        try:
            for rec in stream_records(chunks):
                idx = st["seen"]
                st["seen"] += 1
                text = record_text(rec)
                if text is None or len(text) < MIN_DOC_CHARS:
                    st["skipped"] += 1
                    continue
                is_val = zlib.crc32(f"{name}:{idx}".encode()) % VAL_MOD == 0
                if is_val and st["va"] >= val_q[name]:
                    continue
                if not is_val and st["tr"] >= train_q[name]:
                    if st["va"] >= val_q[name]:
                        break
                    continue
                cap = VAL_MAX_WIN_PER_DOC if is_val else MAX_WIN_PER_DOC
                pending.append((is_val, text[:cap * CHARS_PER_WINDOW_CAP]))
                if len(pending) >= ENCODE_BATCH:
                    flush()
                if st["seen"] % 2000 == 0:
                    log(f"[{name}] docs seen={st['seen']} train_windows={st['tr']}/{train_q[name]} "
                        f"val_windows={st['va']}/{val_q[name]}")
        finally:
            if hasattr(chunks, "close"):
                chunks.close()
        flush()
        log(f"[{name}] done: docs={st['seen']} (skipped {st['skipped']}) train={st['tr']} val={st['va']} "
            f"in {time.time() - t0:.0f}s")
        if st["tr"] < 0.9 * train_q[name]:
            log(f"  warning: [{name}] only produced {st['tr']}/{train_q[name]} train windows (not enough long docs)")
    if not train_out or not val_out:
        raise RuntimeError("no windows were built - check the data paths")
    return np.stack(train_out), np.stack(val_out)


def get_windows(tok):
    sig = {"kind": "sliding_lm", "context": CONTEXT_LENGTH, "stride": STRIDE, "train_windows": MAX_WINDOWS,
           "val_windows": VAL_WINDOWS, "mix": MIX, "vocab": tok.vocab_size, "tok": tok.fingerprint,
           "min_chars": MIN_DOC_CHARS, "max_win": MAX_WIN_PER_DOC, "val_max_win": VAL_MAX_WIN_PER_DOC,
           "val_mod": VAL_MOD}
    for d in walk_find("windows_manifest.json", [CACHE_DIR, INPUT_ROOT]):
        try:
            with open(os.path.join(d, "windows_manifest.json")) as f:
                man = json.load(f)
            if man.get("sig") != sig:
                continue
            tr = np.load(os.path.join(d, "train_windows.npy"))
            va = np.load(os.path.join(d, "val_windows.npy"))
        except Exception as e:
            log(f"cache in {d} unusable ({e})")
            continue
        log(f"loaded cached windows from {d}: train={tr.shape} val={va.shape}")
        if os.path.abspath(d) != os.path.abspath(CACHE_DIR):
            save_windows(tr, va, sig)
        return tr, va
    log("no window cache found, tokenizing the corpus (one-off, cached afterwards)")
    tr, va = build_windows(tok)
    save_windows(tr, va, sig)
    log(f"built windows: train={tr.shape} val={va.shape}")
    return tr, va


def save_windows(tr, va, sig):
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(os.path.join(CACHE_DIR, "train_windows.npy"), tr)
    np.save(os.path.join(CACHE_DIR, "val_windows.npy"), va)
    with open(os.path.join(CACHE_DIR, "windows_manifest.json"), "w") as f:  # written last, means the cache is complete
        json.dump({"sig": sig, "train": len(tr), "val": len(va)}, f)


# model
def padded_vocab(n):
    return ((n + VOCAB_PAD_MULTIPLE - 1) // VOCAB_PAD_MULTIPLE) * VOCAB_PAD_MULTIPLE


@dataclass
class ModelConfig:
    vocab_size: int = padded_vocab(VOCAB_SIZE)      # embedding rows (padded)
    real_vocab: int = VOCAB_SIZE                    # ids >= real_vocab never occur, they're banned when decoding
    d_model: int = D_MODEL
    n_layers: int = N_LAYERS
    n_heads: int = N_HEADS
    n_kv_heads: int = N_KV_HEADS
    ffn_hidden: int = FFN_HIDDEN
    dropout: float = DROPOUT
    max_len: int = CONTEXT_LENGTH + 64
    mtp_heads: int = MTP_HEADS
    mtp_weights: list = field(default_factory=lambda: list(MTP_WEIGHTS))
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    bos_id: int = 3
    mask_id: int = 4


def make_config(tok):
    return ModelConfig(vocab_size=padded_vocab(tok.vocab_size), real_vocab=tok.vocab_size, pad_id=tok.pad_id,
                       unk_id=tok.unk_id, eos_id=tok.eos_id, bos_id=tok.bos_id, mask_id=tok.mask_id)


class WeightedNorm(nn.Module):
    # layernorm with a learned per channel weight and bias: y = w * (x - mean) / sqrt(var + eps) + b
    # replaces rmsnorm (which only rescales). stats and the affine part are done in fp32 and cast back, so it is safe with fp16 autocast

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        return F.layer_norm(x.float(), x.shape[-1:], self.weight.float(), self.bias.float(), self.eps).to(x.dtype)

def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)


def apply_rope(x, cos, sin):
    cos = cos.to(dtype=x.dtype)
    sin = sin.to(dtype=x.dtype)

    x1, x2 = x.chunk(2, dim=-1)

    return torch.cat([
        x1 * cos - x2 * sin,
        x1 * sin + x2 * cos
    ], dim=-1)

def repeat_kv(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd), query head h uses kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQAttention(nn.Module):
    # causal grouped query attention with rope (10 query heads share 2 kv heads)

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

    def forward_cached(self, x, cos, sin, k_cache, v_cache, start, attn_mask):
        # the T new tokens go into cache slots [start, start+T) and attend to slots [0, start+T)
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k_cache[:, :, start:start + T] = k.to(k_cache.dtype)
        v_cache[:, :, start:start + T] = v.to(v_cache.dtype)
        keys = repeat_kv(k_cache[:, :, :start + T], self.groups).to(q.dtype)
        vals = repeat_kv(v_cache[:, :, :start + T], self.groups).to(q.dtype)
        out = F.scaled_dot_product_attention(q, keys, vals, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = WeightedNorm(cfg.d_model)       # applied after the attention residual add
        self.attn = GQAttention(cfg)
        self.norm2 = WeightedNorm(cfg.d_model)       # applied after the ffn residual add
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = self.norm1(x + self.drop(self.attn(x, cos, sin)))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x

    def forward_cached(self, x, cos, sin, cache, start, attn_mask):
        x = self.norm1(x + self.attn.forward_cached(x, cos, sin, cache["k"], cache["v"], start, attn_mask))
        x = self.norm2(x + self.ffn(x))
        return x


class MTPHead(nn.Module):
    # residual projection + weightednorm: z = WeightedNorm(x + W x), the logits are z @ E^T with the shared tied embedding E
    # (x is the output of the last post-norm block so it is already normalised, W starts at zero so every head starts as
    # WeightedNorm(x) and then specialises)

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.norm = WeightedNorm(dim)

    def forward(self, x):
        return self.norm(x + self.proj(x))


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the tied output projection of every head
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([MTPHead(cfg.d_model) for _ in range(cfg.mtp_heads)])
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # scale down the residual branch output projections, like gpt2
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)
        # every head starts as the identity branch, so all four heads begin the same and then drift apart
        for h in self.heads:
            nn.init.zeros_(h.proj.weight)
        self._ban = None

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _rope(self, T, pos_ids=None):
        if pos_ids is None:
            return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        return self.rope_cos[pos_ids][:, None], self.rope_sin[pos_ids][:, None]   # (B, 1, T, hd/2)

    def hidden(self, idx):
        # idx (B, T) -> residual stream (B, T, D) that all mtp heads read
        cos, sin = self._rope(idx.size(1))
        x = self.drop(self.embed(idx))
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return x

    def head_z(self, x, k):
        # normalised features of mtp head k (0 based, head 0 = next token)
        return self.heads[k](x)

    def forward(self, idx):
        # next token logits (B, T, V) from head 1, this is the path decoding uses (training goes through hidden() + the mtp heads)
        return F.linear(self.head_z(self.hidden(idx), 0), self.embed.weight)

    def ban_tensor(self, device):
        # ids that must never be generated: the 5 specials and the padding rows of the embedding
        if self._ban is None or self._ban.device != device:
            c = self.cfg
            ids = sorted({c.pad_id, c.unk_id, c.eos_id, c.bos_id, c.mask_id} | set(range(c.real_vocab, c.vocab_size)))
            self._ban = torch.tensor(ids, dtype=torch.long, device=device)
        return self._ban

    # kv cached generation (head 1 only)
    def init_cache(self, B, total_len, dtype, device):
        kv, hd = self.cfg.n_kv_heads, self.cfg.d_model // self.cfg.n_heads
        return [{"k": torch.zeros(B, kv, total_len, hd, dtype=dtype, device=device),
                 "v": torch.zeros(B, kv, total_len, hd, dtype=dtype, device=device)}
                for _ in self.blocks]

    def forward_cached(self, idx, pos_ids, cache, start, attn_mask):
        # idx (B, T) new tokens with rope positions pos_ids (B, T), returns the next token logits of the last one
        cos, sin = self._rope(idx.size(1), pos_ids)
        x = self.embed(idx)
        for blk, c in zip(self.blocks, cache):
            x = blk.forward_cached(x, cos, sin, c, start, attn_mask)
        return F.linear(self.head_z(x[:, -1], 0), self.embed.weight)


def count_params(model):
    return sum(p.numel() for p in model.parameters())      # tied weights are counted once


# multi token prediction loss
def _ce_sum(z, weight, y):
    return F.cross_entropy(F.linear(z, weight).float(), y, ignore_index=IGNORE, reduction="sum")


def head_ce(z, weight, y, ckpt):
    # summed ce of z (N, D) against y (N,), N tokens at a time. with ckpt the logits of a chunk aren't kept for backward but recomputed,
    # so at most ~CE_CHUNK x V fp32 numbers exist at any moment
    total = z.new_zeros((), dtype=torch.float32)
    for s in range(0, z.size(0), CE_CHUNK):
        zc, yc = z[s:s + CE_CHUNK], y[s:s + CE_CHUNK]
        if ckpt and torch.is_grad_enabled():
            total = total + checkpoint(_ce_sum, zc, weight, yc, use_reentrant=False)
        else:
            total = total + _ce_sum(zc, weight, yc)
    return total, (y != IGNORE).sum()


def head_sums(model, h, w, ckpt, frac=1.0):
    # per head k=1..K: (sum of ce, number of target tokens). head k at position t predicts w[:, t + k]
    # positions whose target is past the window (and pad targets) are ignored
    cfg = model.cfg
    D = h.size(-1)
    hf = h.reshape(-1, D)
    E = model.embed.weight
    out = []
    for k in range(1, cfg.mtp_heads + 1):
        y = F.pad(w[:, k:], (0, k - 1), value=IGNORE)                 # (B, T), target t+k
        y = y.masked_fill(y == cfg.pad_id, IGNORE).reshape(-1)
        hk = hf
        if k > 1 and frac < 1.0:
            sel = torch.rand(y.numel(), device=y.device) < frac
            hk, y = hf[sel], y[sel]
        out.append(head_ce(model.head_z(hk, k - 1), E, y, ckpt))
    return out


def mtp_loss(model, x, w, ckpt=True):
    # returns (weighted training loss, per head mean ce [detached]), per_head[0] is the normal next token loss
    h = model.hidden(x)
    sums = head_sums(model, h, w, ckpt, frac=MTP_POS_FRAC if model.training else 1.0)
    per = [s / n.clamp(min=1) for s, n in sums]
    total = sum(wk * l for wk, l in zip(model.cfg.mtp_weights, per))
    return total, torch.stack(per).detach()


# training helpers
def make_batch(windows, idx, device):
    w = torch.from_numpy(windows[idx].astype(np.int64)).to(device)          # (B, CONTEXT_LENGTH + 1)
    return w[:, :-1], w


@torch.no_grad()
def evaluate(model, windows, device, bs=16):
    # mean ce of every mtp head over the val windows, element 0 is the next token loss (used to pick best.pt)
    model.eval()
    K = model.cfg.mtp_heads
    tot = np.zeros(K)
    cnt = np.zeros(K)
    for i in range(0, len(windows), bs):
        idx = np.arange(i, min(i + bs, len(windows)))
        x, w = make_batch(windows, idx, device)
        with amp_ctx():
            sums = head_sums(model, model.hidden(x), w, ckpt=False)
        for k, (s, n) in enumerate(sums):
            tot[k] += s.item()
            cnt[k] += n.item()
    return (tot / np.maximum(cnt, 1)).tolist()


def lr_at(step, total_steps):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    prog = min(1.0, (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS))
    return PEAK_LR * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * prog)))


def make_optimizer(model):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if p.ndim >= 2 and "embed" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    groups = [{"params": decay, "weight_decay": WEIGHT_DECAY},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=PEAK_LR, betas=BETAS, eps=1e-8,
                             fused=torch.cuda.is_available())


# decoding (batched, kv cache, left padded prompts), head 1 only
def make_prompt_batch(seqs, pad_id, device):
    # left pad token lists into one tensor, mask is True on real tokens
    width = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), width), pad_id, dtype=torch.long)
    mask = torch.zeros(len(seqs), width, dtype=torch.bool)
    for r, s in enumerate(seqs):
        ids[r, width - len(s):] = torch.tensor(s, dtype=torch.long)
        mask[r, width - len(s):] = True
    return ids.to(device), mask.to(device)


def start_generation(model, prompt, prompt_mask, max_new):
    # prefill the cache with the left padded prompt, returns cache, last position logits, lengths, key validity
    B, W = prompt.shape
    dev = prompt.device
    lengths = prompt_mask.sum(1)                                        # real tokens per row
    pos_ids = (torch.arange(W, device=dev)[None] - (W - lengths)[:, None]).clamp(min=0)
    causal = torch.tril(torch.ones(W, W, dtype=torch.bool, device=dev))
    eye = torch.eye(W, dtype=torch.bool, device=dev)
    # pad rows can still see themselves, otherwise their softmax row is empty (nan)
    mask = (causal[None, None] & prompt_mask[:, None, None, :]) | eye[None, None]
    valid = torch.cat([prompt_mask, torch.ones(B, max_new, dtype=torch.bool, device=dev)], dim=1)
    cdtype = AMP["dtype"] if AMP["enabled"] else torch.float32
    cache = model.init_cache(B, W + max_new, cdtype, dev)
    logits = model.forward_cached(prompt, pos_ids, cache, 0, mask)
    return cache, logits, lengths, valid


def next_logits(model, tok, t, W, cache, lengths, valid):
    # feed the token from step t (it sits in cache slot W+t) and get the next logits
    pos = (lengths + t)[:, None]
    mask = valid[:, None, None, :W + t + 1]
    return model.forward_cached(tok[:, None], pos, cache, W + t, mask)


def _expand(x, k):
    return x.repeat_interleave(k, dim=0) if k > 1 else x


def _sample_from(logits, temperature, top_k, top_p):
    logits = logits / temperature
    if top_k > 0:
        kk = min(top_k, logits.size(-1))
        kth = torch.topk(logits, kk).values[:, -1, None]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        probs = sl.softmax(-1)
        drop = (probs.cumsum(-1) - probs) > top_p
        sl = sl.masked_fill(drop, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def _block_ngrams(logits, seq, t, n, pad_id):
    # ban every token that would complete an n-gram already in seq[:, :t], all rows at once
    if n < 2 or t < n:
        return
    s = seq[:, :t]
    ctx = s[:, t - n + 1:]                                   # (R, n-1) the last n-1 tokens
    wins = s.unfold(1, n - 1, 1)[:, : t - n + 1]             # (R, t-n+1, n-1) earlier windows (not the context itself)
    match = (wins == ctx[:, None, :]).all(-1)                # (R, t-n+1)
    nxt = s[:, n - 1:]                                       # token that followed each window
    banned = torch.where(match, nxt, torch.full_like(nxt, pad_id))   # pad is already banned so it's a harmless filler
    logits.scatter_(1, banned, float("-inf"))


@torch.no_grad()
def decode_batch(model, prompt, prompt_mask, max_new=MAX_NEW_TOKENS, mode="beam", beam=4, n=1,
                 temperature=0.8, top_k=0, top_p=1.0, rep_penalty=1.0, no_repeat=0):
    # mode="beam": beam search with `beam` beams (beam=1 is greedy), returns (B, beam, max_new) with the best beam first
    # mode="sample": n independent samples per prompt, returns (B, n, max_new)
    # the prompt is prefilled once and its kv cache is copied to the beams / samples
    # specials and the padding rows of the embedding are banned so every hypothesis has exactly max_new real tokens
    # (same length, so no length penalty needed)
    model.eval()
    B, W = prompt.shape
    dev = prompt.device
    ban = model.ban_tensor(dev)
    K = beam if mode == "beam" else n
    R = B * K
    seq = torch.zeros(R, max_new, dtype=torch.long, device=dev)
    with amp_ctx():
        cache, logits, lengths, valid = start_generation(model, prompt, prompt_mask, max_new)
        cache = [{"k": _expand(c["k"], K), "v": _expand(c["v"], K)} for c in cache]
        logits, lengths, valid = _expand(logits, K), _expand(lengths, K), _expand(valid, K)
        if mode == "beam":
            beam_scores = torch.zeros(B, K, device=dev)
            beam_scores[:, 1:] = -1e9                         # step 0: all beams are the same, only keep beam 0
            base = (torch.arange(B, device=dev) * K)[:, None]
            ident = torch.arange(R, device=dev)
        for t in range(max_new):
            logits = logits.float()
            logits.index_fill_(1, ban, float("-inf"))
            if rep_penalty != 1.0 and t > 0:
                prev = seq[:, :t]
                sc = logits.gather(1, prev)
                sc = torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty)
                logits.scatter_(1, prev, sc)
            _block_ngrams(logits, seq, t, no_repeat, model.cfg.pad_id)
            if mode == "beam":
                V = logits.size(-1)
                logp = torch.log_softmax(logits, dim=-1)
                cand = (beam_scores.view(R, 1) + logp).view(B, K * V)
                top, idx = cand.topk(K, dim=1)
                parent = torch.div(idx, V, rounding_mode="floor")
                tok = idx % V
                flat_parent = (base + parent).view(-1)
                beam_scores = top
                if not torch.equal(flat_parent, ident):       # beams got reordered, so reorder the history and kv cache too
                    seq = seq.index_select(0, flat_parent)
                    for c in cache:
                        c["k"] = c["k"].index_select(0, flat_parent)
                        c["v"] = c["v"].index_select(0, flat_parent)
                tok = tok.view(-1)
            else:
                tok = _sample_from(logits, temperature, top_k, top_p)
            seq[:, t] = tok
            if t < max_new - 1:
                logits = next_logits(model, tok, t, W, cache, lengths, valid)
    return seq.view(B, K, max_new)


@torch.no_grad()
def self_test(model, val_w, device):
    # the cached + left padded decoding path should give the same result as a plain full forward pass (fp32)
    model.eval()
    saved = AMP["enabled"]
    AMP["enabled"] = False
    try:
        if val_w is not None:
            rows = val_w[:3].astype(np.int64).tolist()
        else:
            rng = np.random.default_rng(0)
            rows = rng.integers(5, model.cfg.real_vocab, size=(3, 800)).tolist()
        seqs = [r[:L] for r, L in zip(rows, (300, 520, 700))]
        ids, mask = make_prompt_batch(seqs, model.cfg.pad_id, device)
        W = ids.size(1)
        steps = 3
        cache, logits, lengths, valid = start_generation(model, ids, mask, steps)
        for t in range(steps):
            ref = torch.stack([model(torch.tensor([s], dtype=torch.long, device=device))[0, -1] for s in seqs])
            err = (logits.float() - ref.float()).abs().max().item()
            if err > 5e-3:
                raise RuntimeError(f"KV-cache self-test failed at step {t}: max logit diff {err:.5f}")
            nxt = ref.argmax(-1)
            seqs = [s + [int(x)] for s, x in zip(seqs, nxt.tolist())]
            if t < steps - 1:
                logits = next_logits(model, nxt, t, W, cache, lengths, valid)
        log(f"self-test passed: cached left-padded decoding matches the full forward pass (last diff {err:.2e})")
    finally:
        AMP["enabled"] = saved


# scoring helpers for val windows
def rouge_l_f1(ref, hyp):
    a, b = ref.split(), hyp.split()
    if not a or not b:
        return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    lcs = prev[-1]
    if lcs == 0:
        return 0.0
    p, r = lcs / len(b), lcs / len(a)
    return 2 * p * r / (p + r)


def split_window(row, tok):
    # cut a val window into prompt | reference so the reference starts on a word boundary
    # with the </w> tokenizer that is where the token before the cut ends a word
    for s in range(PROMPT_TOKENS, PROMPT_TOKENS - 25, -1):
        if tok.is_end[row[s - 1]]:
            return s
    return PROMPT_TOKENS

def val_gen_score(model, tok, val_w, device, n=64, bs=32):
    # mean rouge-l f1 of greedy continuations on the first n val windows (cheap monitor after every epoch)
    scores, samples = [], []
    n = min(n, len(val_w))
    for i in range(0, n, bs):
        rows = val_w[i:min(i + bs, n)].astype(np.int64).tolist()
        splits = [split_window(r, tok) for r in rows]
        ids, mask = make_prompt_batch([r[:s] for r, s in zip(rows, splits)], tok.pad_id, device)
        gen = decode_batch(model, ids, mask, mode="beam", beam=1, no_repeat=NO_REPEAT_NGRAM_V2)[:, 0].cpu().tolist()
        for r, s, g in zip(rows, splits, gen):
            ref, hyp = to_words(r[s:s + MAX_NEW_TOKENS], tok), to_words(g, tok)
            scores.append(rouge_l_f1(ref, hyp))
            samples.append((ref, hyp))
    return float(np.mean(scores)), samples


# decoding v2, same as model 2: beam / greedy with a vectorised no repeat ngram, mbr, bertscore mbr,
# at least 128 words, and a val sweep to pick the recipe
# new in model 3: the chosen recipe and the test generations are saved while running, so a decode run
# that dies (or hits the 12h limit) continues instead of starting over
def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


# mbr: pick the candidate that agrees most with the others
def _grams(words):
    return Counter(words), Counter(zip(words, words[1:]))


def _f1(a, b):
    if not a or not b:
        return 0.0
    ov = sum((a & b).values())
    if ov == 0:
        return 0.0
    p, r = ov / sum(b.values()), ov / sum(a.values())
    return 2 * p * r / (p + r)


def _utility(fa, fb):
    return MBR_W_UNI * _f1(fa[0], fb[0]) + (1.0 - MBR_W_UNI) * _f1(fa[1], fb[1])


def _first_full(cands, tok):
    # first candidate with at least 128 words, else just the first one
    for c in cands:
        if n_words(c, tok) >= TARGET_WORDS:
            return c
    return cands[0]


def _mbr_choose(best_beam, samples, tok):
    # hypotheses = best beam + all samples, pseudo references = the samples
    # (the beams are near duplicates and would just vote for each other)
    # candidates with under 128 words are only used if nothing else exists
    cands = [best_beam] + list(samples)
    texts = [to_words(c, tok) for c in cands]
    words = [t.lower().split() for t in texts]
    full = [i for i, w in enumerate(words) if len(w) >= TARGET_WORDS] or list(range(len(cands)))
    feats = [_grams(w) for w in words]
    refs = list(range(1, len(cands)))
    best_i, best_s = full[0], -1.0
    for i in full:
        others = [j for j in refs if j != i]
        if not others:
            continue
        s = sum(_utility(feats[i], feats[j]) for j in others) / len(others)
        if s > best_s:
            best_i, best_s = i, s
    return cands[best_i]


# mbr with bertscore as the score
class BertScorer:
    # bertscore style greedy token matching (no idf, special tokens have no weight)
    # each text is encoded once, then any two texts in the pool are compared with a matrix product

    def __init__(self, name, layer, device):
        from transformers import AutoModel, AutoTokenizer
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModel.from_pretrained(name)
        model.encoder.layer = nn.ModuleList(model.encoder.layer[:layer])      # bert-score also stops at this layer
        if device.type == "cuda":
            model = model.half()
        self.model = model.to(device).eval()

    @torch.no_grad()
    def embed(self, texts, batch_size=32):
        # one (n_tokens, dim) tensor of unit length vectors per text
        out = []
        for s in range(0, len(texts), batch_size):
            enc = self.tokenizer(texts[s:s + batch_size], return_tensors="pt", padding=True, truncation=True,
                                 max_length=512, return_special_tokens_mask=True)
            special = enc.pop("special_tokens_mask").bool().to(self.device)
            enc = enc.to(self.device)
            hidden = F.normalize(self.model(**enc).last_hidden_state.float(), dim=-1)
            keep = enc["attention_mask"].bool() & ~special
            out.extend(hidden[r][keep[r]] for r in range(hidden.size(0)))
        return out

    @staticmethod
    def pairwise_f1(embs):
        # (n, n) matrix, [i, j] = bertscore f1 between text i and text j
        n = len(embs)
        width = max(1, max(e.size(0) for e in embs))
        grid = embs[0].new_zeros(n, width, embs[0].size(1))
        valid = torch.zeros(n, width, dtype=torch.bool, device=grid.device)
        for i, e in enumerate(embs):
            grid[i, :e.size(0)] = e
            valid[i, :e.size(0)] = True
        sim = torch.einsum("ild,jmd->ijlm", grid, grid)             # cosine similarity of every token pair
        sim.masked_fill_(~valid[None, :, None, :], -2.0)            # padding of text j never matches
        best = sim.max(dim=-1).values                               # (i, j, l): best match in j for token l of i
        sizes = valid.sum(1).clamp(min=1).float()
        prec = (best * valid[:, None, :]).sum(-1) / sizes[:, None]  # i as candidate, j as reference
        return 2 * prec * prec.T / (prec + prec.T).clamp(min=1e-8)  # recall of [i, j] is the precision of [j, i]


_scorer_state = {"scorer": None, "failed": False}


def get_scorer():
    # the shared BertScorer, loaded on first use, None if the embedding model can't be loaded
    st = _scorer_state
    if st["scorer"] is None and not st["failed"]:
        try:
            dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            st["scorer"] = BertScorer(EMBED_MODEL, EMBED_LAYER, dev)
            log(f"MBR scorer ready: {EMBED_MODEL}, layer {EMBED_LAYER}, on {dev}")
        except Exception as e:
            st["failed"] = True
            log(f"  (BERTScore-MBR unavailable: {type(e).__name__}: {str(e)[:80]})")
    return st["scorer"]


def _mbr_choose_bs(best_beam, samples, tok, scorer):
    # like _mbr_choose but agreement between candidates is bertscore f1. the beam is only a candidate (never votes),
    # the samples are candidates and voters. short candidates are only used if nothing else exists
    cands = [best_beam] + list(samples)
    texts = [to_words(c, tok) for c in cands]
    full = [i for i, t in enumerate(texts) if len(t.split()) >= TARGET_WORDS] or list(range(len(cands)))
    f1 = scorer.pairwise_f1(scorer.embed(texts))
    f1.fill_diagonal_(0.0)
    f1[:, 0] = 0.0
    votes = torch.full((len(cands),), float(len(samples)), device=f1.device)
    votes[1:] -= 1                                                  # a sample doesn't vote for itself
    score = (f1.sum(1) / votes.clamp(min=1)).tolist()
    return cands[max(full, key=lambda i: score[i])]


# run one recipe over many prompts
def _decode_group(model, tok, ids, mask, c):
    kind = c["kind"]
    common = dict(max_new=MAX_NEW_TOKENS, rep_penalty=c.get("rep", 1.0), no_repeat=c.get("ng", 0))
    if kind == "sample":
        seqs = decode_batch(model, ids, mask, mode="sample", n=c["n"], temperature=c["temperature"],
                            top_k=c.get("top_k", 0), top_p=c.get("top_p", 1.0), **common).cpu().tolist()
        return [_first_full(s, tok) for s in seqs]
    beams = decode_batch(model, ids, mask, mode="beam", beam=c["beam"], **common).cpu().tolist()
    if kind == "beam":
        return [_first_full(b, tok) for b in beams]
    samples = decode_batch(model, ids, mask, mode="sample", n=c["n"], temperature=c["temperature"],
                           top_k=c.get("top_k", 0), top_p=c.get("top_p", 1.0), **common).cpu().tolist()
    if kind == "mbr_bs":
        scorer = get_scorer()
        return [_mbr_choose_bs(beams[b][0], samples[b], tok, scorer) for b in range(len(beams))]
    return [_mbr_choose(beams[b][0], samples[b], tok) for b in range(len(beams))]


def _safe_batch(fn, items):
    # run fn(items), on cuda oom split the batch in half and retry
    try:
        return fn(items)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(items) == 1:
            raise
        h = len(items) // 2
        log(f"  CUDA OOM, splitting batch of {len(items)}")
        return _safe_batch(fn, items[:h]) + _safe_batch(fn, items[h:])


def _extend_until_target(model, tok, prompt_ids, gen_ids, device):
    # keep generating (beam search) until the continuation has at least 128 words
    for _ in range(EXTEND_ROUNDS):
        if n_words(gen_ids, tok) >= TARGET_WORDS:
            break
        ctx = (prompt_ids + gen_ids)[-PROMPT_TOKENS:]
        ids, mask = make_prompt_batch([ctx], tok.pad_id, device)
        more = decode_batch(model, ids, mask, max_new=EXTEND_TOKENS, mode="beam", beam=BEAM_WIDTH,
                            no_repeat=NO_REPEAT_NGRAM_V2)[0, 0].tolist()
        gen_ids = gen_ids + more
    return gen_ids


def load_partial(sig):
    # generations saved by an earlier run (crashed or timed out) with the same recipe, checkpoint and prompts
    best = {}
    for d in walk_find("test_gen_partial.json", [CKPT_DIR, INPUT_ROOT]):
        data = read_json(os.path.join(d, "test_gen_partial.json"))
        if data and data.get("sig") == sig and len(data.get("done", {})) > len(best):
            best = data["done"]
    return {int(k): v for k, v in best.items()}


def run_recipe(model, tok, enc, c, device, name="", partial_sig=None):
    # enc is a list of prompt token id lists, returns the chosen continuation token ids for each prompt (same order)
    # with partial_sig the finished prompts get written to CKPT_DIR/test_gen_partial.json while running
    order = sorted(range(len(enc)), key=lambda i: len(enc[i]))      # group similar lengths, less padding
    rows_pp = c["beam"] if c["kind"] == "beam" else (c["n"] if c["kind"] == "sample" else max(c["beam"], c["n"]))
    bs = max(1, min(32, ROW_BUDGET // rows_pp))
    out = [None] * len(enc)
    partial_path = os.path.join(CKPT_DIR, "test_gen_partial.json")
    if partial_sig:
        for i, g in load_partial(partial_sig).items():
            if i < len(out):
                out[i] = g
        n_have = sum(o is not None for o in out)
        if n_have:
            log(f"  [{name}] continuing: {n_have}/{len(enc)} prompts were already decoded by an earlier run")
    todo = [i for i in order if out[i] is None]

    def work(idxs):
        ids, mask = make_prompt_batch([enc[i] for i in idxs], tok.pad_id, device)
        return list(zip(idxs, _decode_group(model, tok, ids, mask, c)))

    def flush_partial():
        atomic_json({"sig": partial_sig, "done": {str(i): g for i, g in enumerate(out) if g is not None}}, partial_path)

    t0, last_flush = time.time(), time.time()
    for s in range(0, len(todo), bs):
        for i, g in _safe_batch(work, todo[s:s + bs]):
            out[i] = g
        done = min(s + bs, len(todo))
        if partial_sig and time.time() - last_flush > PARTIAL_SAVE_SEC:
            flush_partial()
            last_flush = time.time()
        if (s // bs) % 10 == 0 or done == len(todo):
            el = time.time() - t0
            log(f"  [{name}] {done}/{len(todo)} prompts | {fmt_time(el)} elapsed | eta {fmt_time(el / done * (len(todo) - done))}")
    if partial_sig and todo:
        flush_partial()
    n_short = 0
    for i in range(len(out)):
        if n_words(out[i], tok) < TARGET_WORDS:
            n_short += 1
            out[i] = _extend_until_target(model, tok, enc[i], out[i], device)
    if n_short:
        log(f"  [{name}] {n_short} continuations were < {TARGET_WORDS} words and were extended")
    return out


# configs and val sweep
def recipe_table(with_bs=True):
    ng = NO_REPEAT_NGRAM_V2
    table = {
        "baseline_sample": dict(kind="sample", n=1, temperature=0.5, top_k=40, top_p=0.9, rep=1.1, ng=3),  # old model 2 recipe
        "greedy": dict(kind="beam", beam=1, ng=ng),
        "beam": dict(kind="beam", beam=BEAM_WIDTH, ng=ng),
        "beam8": dict(kind="beam", beam=8, ng=ng),
        "mbr": dict(kind="mbr", beam=BEAM_WIDTH, n=MBR_SAMPLES, temperature=MBR_TEMPERATURE,
                    top_k=0, top_p=MBR_TOP_P, rep=1.0, ng=ng),
    }
    if with_bs and get_scorer() is not None:
        for name, temp in (("mbr_bs", 0.7), ("mbr_bs_hot", 1.0)):
            table[name] = dict(kind="mbr_bs", beam=BEAM_WIDTH, n=BS_MBR_SAMPLES, temperature=temp,
                               top_k=0, top_p=MBR_TOP_P, rep=1.0, ng=ng)
    return table


def bertscore_f1(hyps, refs, device):
    # mean bertscore f1 if the bert-score package (and its model) loads, else None
    try:
        try:
            import bert_score  # noqa: F401
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=False)
        from bert_score import score as bs_score
        _, _, f = bs_score(hyps, refs, lang="en", device=str(device), batch_size=16, verbose=False)
        return float(f.mean())
    except Exception as e:
        log(f"  (BERTScore unavailable: {type(e).__name__}: {str(e)[:80]})")
        return None


def val_sweep(model, tok, val_w, device, n):
    idx = np.linspace(0, len(val_w) - 1, min(n, len(val_w))).astype(int)      # spread over all four sources
    rows = val_w[idx].astype(np.int64).tolist()
    splits = [split_window(r, tok) for r in rows]
    enc = [r[:s] for r, s in zip(rows, splits)]
    refs = [to_words(r[s:s + MAX_NEW_TOKENS], tok) for r, s in zip(rows, splits)]
    results = {}
    for name, c in recipe_table().items():
        t0 = time.time()
        gens = run_recipe(model, tok, enc, c, device, name)
        hyps = [to_words(g, tok) for g in gens]
        rl = float(np.mean([rouge_l_f1(r, h) for r, h in zip(refs, hyps)]))
        ng = float(np.mean([_utility(_grams(r.lower().split()), _grams(h.lower().split())) for r, h in zip(refs, hyps)]))
        bs = bertscore_f1(hyps, refs, device)
        wc = float(np.mean([len(h.split()) for h in hyps]))
        results[name] = {"bertscore": bs, "ngram_f1": ng, "rougeL": rl, "avg_words": wc, "sec": time.time() - t0}
        log(f"SWEEP {name:>16s} | BERTScore {('%.4f' % bs) if bs is not None else '  n/a '} | ngram-F1 {ng:.4f} | "
            f"ROUGE-L {rl:.4f} | words {wc:.1f} | {time.time() - t0:.0f}s")
    key = (lambda k: results[k]["bertscore"]) if all(v["bertscore"] is not None for v in results.values()) \
        else (lambda k: results[k]["ngram_f1"])
    best = max(results, key=key)
    return best, results


def load_cached_val_windows():
    # val windows from the training run's cache (never rebuilds the corpus), None if not found
    for d in walk_find("val_windows.npy", [CACHE_DIR, INPUT_ROOT]):
        try:
            va = np.load(os.path.join(d, "val_windows.npy"))
            if va.ndim == 2 and va.shape[1] == CONTEXT_LENGTH + 1:
                log(f"validation windows loaded from {d}: {va.shape}")
                return va
        except Exception as e:
            log(f"could not read val windows in {d}: {e}")
    return None


# predict on the test set
def load_test_prompts():
    # test file -> {id: {"text": prompt}}, like the starter code
    token = hf_token()
    test_file_path = None
    for name in (TEST_FILE, TEST_FILE_ALT):
        try:
            test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=name,
                                             repo_type="dataset", token=token)
            break
        except Exception as e:
            log(f"could not download {name}: {type(e).__name__}")
    if test_file_path is None:               # last resort: any json in the Test/ folder
        found = sorted(f for f in list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=token)
                       if f.startswith("Phase_1_Pretraining/Test/") and f.endswith(".json"))
        if not found:
            raise FileNotFoundError("no test json found in the Hugging Face dataset")
        test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=found[0],
                                         repo_type="dataset", token=token)
    log(f"Test file: {test_file_path}")
    with open(test_file_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    log(f"Number of test examples: {len(test_data)}")
    return test_data


def choose_recipe(model, tok, val_w, device, step):
    # returns (name, recipe), 'auto' sweeps the recipes on val windows unless this checkpoint was already swept
    if DECODE_MODE != "auto" or val_w is None:
        name = "beam" if DECODE_MODE == "auto" else DECODE_MODE
        if DECODE_MODE == "auto":
            log("no validation windows found -> falling back to plain beam search")
        table = recipe_table(with_bs=name.startswith("mbr_bs"))
        if name not in table:
            log(f"recipe {name} unavailable -> beam")
            name = "beam"
        log(f"recipe: {name}")
        return name, table[name]
    if not QUICK_TEST:
        for d in walk_find("decoding_sweep_v2.json", [CKPT_DIR, INPUT_ROOT]):
            saved = read_json(os.path.join(d, "decoding_sweep_v2.json"))
            if saved and saved.get("tag") == MODEL_TAG and saved.get("model_step") == step and saved.get("chosen"):
                name = saved["chosen"]
                table = recipe_table(with_bs=name.startswith("mbr_bs"))
                if name in table:
                    log(f"re-using the recipe chosen for this checkpoint earlier: {name} (from {d})")
                    return name, table[name]
    best, results = val_sweep(model, tok, val_w, device, 32 if QUICK_TEST else VAL_SWEEP_N)
    atomic_json({"tag": MODEL_TAG, "model_step": step, "chosen": best, "results": results},
                os.path.join(CKPT_DIR, "decoding_sweep_v2.json"))
    log(f"chosen recipe: {best}")
    return best, recipe_table()[best]


def predict_v2(model, tok, val_w, device):
    prompts = load_test_prompts()
    keys = list(prompts.keys())
    if QUICK_TEST:
        keys = keys[:16]
    step = LOADED.get("global_step", -1)
    name, recipe = choose_recipe(model, tok, val_w, device, step)
    enc = []
    for k in keys:
        ids = tok.encode(prompts[k]["text"])[-PROMPT_TOKENS:]
        enc.append(ids if ids else [tok.eos_id])
    sig = None
    if not QUICK_TEST:
        digest = hashlib.md5("\x00".join(str(k) for k in keys).encode() + str(sum(map(sum, enc))).encode()).hexdigest()[:12]
        sig = f"{MODEL_TAG}|{name}|{step}|{len(enc)}|{digest}"
    gens = run_recipe(model, tok, enc, recipe, device, "test", partial_sig=sig)
    preds = {str(k): {"text": to_words(g, tok)} for k, g in zip(keys, gens)}
    short = sum(1 for v in preds.values() if len(v["text"].split()) < TARGET_WORDS)
    lens = [len(v["text"].split()) for v in preds.values()]
    log(f"prediction word counts: min/avg/max = {min(lens)}/{np.mean(lens):.1f}/{max(lens)} | below {TARGET_WORDS}: {short}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if QUICK_TEST:
        path = os.path.join(OUTPUT_DIR, "prediction_QUICKTEST.json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump(preds, f, ensure_ascii=False, indent=2)
        log(f"quick test only - wrote {path}, no submission zip")
        return
    assert set(preds) == {str(k) for k in prompts}, "prediction ids differ from test ids"
    preds = {str(k): preds[str(k)] for k in prompts}                       # keep the official id order
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(PREDICTION_PATH, arcname="prediction.json")                # has to be at the root of the zip
    log(f"saved {PREDICTION_PATH} and {SUBMISSION_ZIP} ({len(preds)} predictions)")


# checkpoints: find, resume, carry over
LOADED = {}      # which weights were used for prediction: {"source", "global_step", ...}


def find_previous():
    # newest complete latest.pt (by global_step) in our dir or any attached kaggle input
    best = None
    for d in walk_find("latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "latest_meta.json"))
        if meta is None or meta.get("tag") != MODEL_TAG or not os.path.exists(os.path.join(d, "latest.pt")):
            continue
        if best is None or meta["global_step"] > best[1]["global_step"]:
            best = (d, meta)
    return best


def find_best_weights():
    # best.pt (lowest val loss) of this model, in our dir or any attached kaggle input
    best = None
    for d in walk_find("best_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "best_meta.json"))
        if not meta or meta.get("tag") != MODEL_TAG or not os.path.exists(os.path.join(d, "best.pt")):
            continue
        if best is None or meta["val_loss"] < best[1]["val_loss"]:
            best = (d, meta)
    return best


def carry_over(src_dir):
    # copy the best checkpoint + log from an older run's output into this run's checkpoint dir
    if os.path.abspath(src_dir) == os.path.abspath(CKPT_DIR):
        return
    os.makedirs(CKPT_DIR, exist_ok=True)
    old_best = read_json(os.path.join(src_dir, "best_meta.json"))
    cur_best = read_json(os.path.join(CKPT_DIR, "best_meta.json"))
    if old_best and os.path.exists(os.path.join(src_dir, "best.pt")):
        if cur_best is None or old_best["val_loss"] < cur_best["val_loss"]:
            shutil.copy(os.path.join(src_dir, "best.pt"), os.path.join(CKPT_DIR, "best.pt.tmp"))
            os.replace(os.path.join(CKPT_DIR, "best.pt.tmp"), os.path.join(CKPT_DIR, "best.pt"))
            shutil.copy(os.path.join(src_dir, "best_meta.json"), os.path.join(CKPT_DIR, "best_meta.json"))
    src_log, dst_log = os.path.join(src_dir, "train_log.csv"), os.path.join(CKPT_DIR, "train_log.csv")
    if os.path.exists(src_log) and not os.path.exists(dst_log):
        shutil.copy(src_log, dst_log)


def load_weights_for_prediction(model):
    best = find_best_weights()
    if best is not None:
        d, meta = best
        ckpt = torch.load(os.path.join(d, "best.pt"), map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        LOADED.update(source="best", global_step=meta["global_step"], val_loss=meta["val_loss"])
        log(f"predicting with best.pt from {d}: next-token val loss {meta['val_loss']:.4f} "
            f"(epoch {meta['epoch'] + 1}, global step {meta['global_step']})")
        return
    prev = find_previous()
    if prev is None:
        raise FileNotFoundError("no best.pt or latest.pt of this model found - attach the previous run's output")
    ckpt = torch.load(os.path.join(prev[0], "latest.pt"), map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    LOADED.update(source="latest", global_step=prev[1]["global_step"])
    log(f"no best.pt found, predicting with latest.pt from {prev[0]}")


LOG_FIELDS = ["time", "kind", "epoch", "step_in_epoch", "global_step", "train_loss", "train_ntp", "val_loss",
              "val_ppl", "val_heads", "lr", "rouge_l", "elapsed_h"]


def append_log(row):
    path = os.path.join(CKPT_DIR, "train_log.csv")
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new:
            wr.writeheader()
        wr.writerow({k: row.get(k, "") for k in LOG_FIELDS})


# training loop
def train(model, tok, train_w, val_w, device, t_start):
    cfg = model.cfg
    K = cfg.mtp_heads
    os.makedirs(CKPT_DIR, exist_ok=True)
    n_train = len(train_w)
    steps_per_epoch = n_train // EFF_BS
    total_steps = EPOCHS * steps_per_epoch
    opt = make_optimizer(model)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP["enabled"] and AMP["dtype"] == torch.float16)
    state = {"epoch": 0, "step_in_epoch": 0, "global_step": 0, "best_val": float("inf")}

    def save_latest(epoch, step_in_epoch):
        payload = {"model": model.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),
                   "epoch": epoch, "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                   "best_val": state["best_val"], "config": asdict(cfg), "tag": MODEL_TAG}
        atomic_torch_save(payload, os.path.join(CKPT_DIR, "latest.pt"))
        atomic_json({"epoch": epoch, "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                     "best_val": state["best_val"], "tag": MODEL_TAG, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
                    os.path.join(CKPT_DIR, "latest_meta.json"))

    prev = find_previous()
    if prev is not None:
        d, meta = prev
        log(f"resuming from {d}: epoch {meta['epoch'] + 1}, step {meta['step_in_epoch']}, global step {meta['global_step']}")
        ckpt = torch.load(os.path.join(d, "latest.pt"), map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        state.update({k: ckpt[k] for k in state})
        del ckpt
        carry_over(d)
        if os.path.abspath(d) != os.path.abspath(CKPT_DIR):
            save_latest(state["epoch"], state["step_in_epoch"])     # so this session's output is complete from the start
    else:
        log("no previous checkpoint found, starting from scratch")
    best_meta = read_json(os.path.join(CKPT_DIR, "best_meta.json"))
    if best_meta:
        state["best_val"] = min(state["best_val"], best_meta["val_loss"])

    if state["epoch"] >= EPOCHS:
        log("all epochs already finished in an earlier session")
        return True

    def run_validation(epoch, step_in_epoch, last_loss):
        vls = evaluate(model, val_w, device)
        vl = vls[0]                                              # next token loss (head 1), comparable with model 2
        improved = vl < state["best_val"]
        if improved:
            state["best_val"] = vl
            atomic_torch_save({"model": model.state_dict(), "val_loss": vl, "epoch": epoch,
                               "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                               "config": asdict(cfg), "tag": MODEL_TAG}, os.path.join(CKPT_DIR, "best.pt"))
            atomic_json({"tag": MODEL_TAG, "val_loss": vl, "val_ppl": math.exp(vl), "val_heads": vls, "epoch": epoch,
                         "step_in_epoch": step_in_epoch, "global_step": state["global_step"]},
                        os.path.join(CKPT_DIR, "best_meta.json"))
        tag = "(NEW BEST, saved best.pt)" if improved else "(best so far %.4f)" % state["best_val"]
        heads = " / ".join(f"{v:.3f}" for v in vls)
        log(f"  >> validation: next-token loss {vl:.4f} ppl {math.exp(vl):.2f} {tag} | heads t+1..t+{K}: {heads}")
        append_log({"time": time.strftime("%H:%M:%S"), "kind": "val", "epoch": epoch + 1,
                    "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                    "train_loss": f"{last_loss:.4f}", "val_loss": f"{vl:.4f}", "val_ppl": f"{math.exp(vl):.2f}",
                    "val_heads": heads, "elapsed_h": f"{(time.time() - t_start) / 3600:.2f}"})
        model.train()
        return vl

    def out_of_time():
        return (time.time() - t_start) > MAX_RUNTIME_H * 3600

    session_first_step = state["global_step"]
    session_t0 = time.time()
    log(f"steps/epoch={steps_per_epoch} total steps={total_steps} effective batch={EFF_BS} windows "
        f"({EFF_BS * CONTEXT_LENGTH:,} tokens/step) | MTP weights {cfg.mtp_weights}, positions for heads 2..{K}: "
        f"{MTP_POS_FRAC:.0%}")

    model.train()
    bad_streak = 0
    for epoch in range(state["epoch"], EPOCHS):
        perm = np.random.default_rng(SEED + epoch).permutation(n_train)   # same order after resuming
        first = state["step_in_epoch"] if epoch == state["epoch"] else 0
        done = first
        run_loss, run_ntp, run_n, last_loss = 0.0, 0.0, 0, float("nan")
        t_log = time.time()
        try:
            for step in range(first, steps_per_epoch):
                lr = lr_at(state["global_step"], total_steps)
                for g in opt.param_groups:
                    g["lr"] = lr
                batch_ids = perm[step * EFF_BS:(step + 1) * EFF_BS]
                stats = torch.zeros(1 + K, device=device)             # [weighted loss, CE of head 1..K]
                for m in range(ACCUM):
                    idx = batch_ids[m * MICRO_BS:(m + 1) * MICRO_BS]
                    x, w = make_batch(train_w, idx, device)
                    with amp_ctx():
                        loss, per = mtp_loss(model, x, w, ckpt=True)
                    scaler.scale(loss / ACCUM).backward()
                    stats[0] += loss.detach() / ACCUM
                    stats[1:] += per / ACCUM
                step_loss, step_ntp = stats[0].item(), stats[1].item()
                if math.isfinite(step_loss):
                    scaler.unscale_(opt)
                    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    if torch.isfinite(gnorm):
                        scaler.step(opt)
                        bad_streak = 0
                    else:
                        bad_streak += 1                              # fp16 overflow while the scaler is still calibrating, skip this step
                    scaler.update()
                else:
                    gnorm = torch.tensor(float("nan"))
                    bad_streak += 1
                    log(f"  warning: non-finite loss at step {done + 1}, update skipped")
                opt.zero_grad(set_to_none=True)
                if bad_streak > 100:
                    raise RuntimeError("100 consecutive skipped updates: training has diverged")

                state["global_step"] += 1
                done = step + 1
                if math.isfinite(step_loss):
                    run_loss += step_loss
                    run_ntp += step_ntp
                    run_n += 1
                    last_loss = step_ntp

                if state["global_step"] % LOG_EVERY == 0 or done == steps_per_epoch:
                    now = time.time()
                    sess_steps = state["global_step"] - session_first_step
                    sec_per_step = (now - session_t0) / max(sess_steps, 1)
                    remaining = total_steps - state["global_step"]
                    avg, avg_ntp = run_loss / max(run_n, 1), run_ntp / max(run_n, 1)
                    overall = 100.0 * state["global_step"] / total_steps
                    heads = " ".join(f"{v:.2f}" for v in stats[1:].tolist())
                    log(f"[ep {epoch + 1}/{EPOCHS}] step {done}/{steps_per_epoch} ({100 * done / steps_per_epoch:.1f}%) | "
                        f"overall {overall:.1f}% | mtp {avg:.4f} | ntp {avg_ntp:.4f} (heads {heads}) | "
                        f"gnorm {float(gnorm):.2f} | lr {lr:.2e} | "
                        f"{EFF_BS * LOG_EVERY / max(now - t_log, 1e-9):.1f} win/s | "
                        f"elapsed {fmt_time(now - t_start)} | ETA {fmt_time(remaining * sec_per_step)}")
                    atomic_json({"epoch": epoch + 1, "epochs": EPOCHS, "step_in_epoch": done,
                                 "steps_per_epoch": steps_per_epoch, "global_step": state["global_step"],
                                 "total_steps": total_steps, "overall_percent": round(overall, 2),
                                 "train_loss": avg, "train_ntp_loss": avg_ntp, "best_val_loss": state["best_val"],
                                 "eta_this_session_h": round(remaining * sec_per_step / 3600, 2),
                                 "updated": time.strftime("%Y-%m-%d %H:%M:%S")},
                                os.path.join(CKPT_DIR, "progress.json"))
                    append_log({"time": time.strftime("%H:%M:%S"), "kind": "train", "epoch": epoch + 1,
                                "step_in_epoch": done, "global_step": state["global_step"],
                                "train_loss": f"{avg:.4f}", "train_ntp": f"{avg_ntp:.4f}", "lr": f"{lr:.3e}",
                                "elapsed_h": f"{(now - t_start) / 3600:.2f}"})
                    run_loss, run_ntp, run_n, t_log = 0.0, 0.0, 0, now

                if done < steps_per_epoch:
                    if state["global_step"] % EVAL_EVERY == 0:
                        run_validation(epoch, done, last_loss)
                    if state["global_step"] % SAVE_EVERY == 0:
                        save_latest(epoch, done)
                        log(f"  >> saved latest.pt (epoch {epoch + 1}, step {done})")
                    if out_of_time():
                        save_latest(epoch, done)
                        log(f"  >> time budget used up, saved latest.pt at epoch {epoch + 1} step {done}. "
                            f"Re-run in a new session to continue.")
                        return False
        except BaseException as e:           # on a crash (oom, ctrl-c...) keep everything up to the last finished step
            log(f"  !! training interrupted by {type(e).__name__}: {str(e)[:120]} - saving latest.pt at "
                f"epoch {epoch + 1} step {done}")
            try:
                save_latest(epoch, done)
            except Exception as e2:
                log(f"  !! emergency save failed: {type(e2).__name__}: {e2}")
            raise

        # end of epoch
        log(f"===== epoch {epoch + 1}/{EPOCHS} finished =====")
        run_validation(epoch, steps_per_epoch, last_loss)
        save_latest(epoch + 1, 0)
        log(f"  >> epoch checkpoint saved (latest.pt now points to start of epoch {epoch + 2})")
        try:     # if this monitor fails, don't kill the training run
            score, samples = val_gen_score(model, tok, val_w, device, n=32)
            log(f"  >> greedy+ngram{NO_REPEAT_NGRAM_V2} ROUGE-L on 32 val windows: {score:.4f}")
            log(f"  >> sample reference : {samples[0][0][:300]}")
            log(f"  >> sample generated : {samples[0][1][:300]}")
            append_log({"time": time.strftime("%H:%M:%S"), "kind": "gen", "epoch": epoch + 1,
                        "global_step": state["global_step"], "rouge_l": f"{score:.4f}",
                        "elapsed_h": f"{(time.time() - t_start) / 3600:.2f}"})
        except Exception as e:
            log(f"  >> generation monitor skipped: {type(e).__name__}: {e}")
        model.train()
        state["epoch"], state["step_in_epoch"] = epoch + 1, 0
        if epoch + 1 < EPOCHS and out_of_time():
            log("  >> time budget used up after this epoch; re-run in a new session to continue.")
            return False
    return True


# main
def setup_device():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        AMP["enabled"] = True
        AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
    else:
        log("WARNING: no GPU visible, this will be extremely slow")
    return device


def build_model(tok, device):
    cfg = make_config(tok)
    model = DecoderLM(cfg).to(device)
    n_params = count_params(model)
    log(f"model parameters: {n_params / 1e6:.1f}M (embedding rows {cfg.vocab_size}, {cfg.mtp_heads} MTP heads)")
    assert 50e6 <= n_params <= 300e6, "parameter count outside the 50M-300M limit"
    return model


def main():
    # train (resumes from the newest latest.pt) and predict once all epochs are done
    t_start = time.time()
    hf_login()
    os.makedirs(CKPT_DIR, exist_ok=True)
    device = setup_device()
    tok = load_tokenizer()
    train_w, val_w = get_windows(tok)
    log(f"train windows {len(train_w):,} ({len(train_w) * CONTEXT_LENGTH / 1e6:.0f}M tokens/epoch) | val windows {len(val_w):,}")
    model = build_model(tok, device)
    if RUN_SELF_TEST:
        self_test(model, val_w, device)
    finished = train(model, tok, train_w, val_w, device, t_start)
    if finished and RUN_PREDICTION:
        load_weights_for_prediction(model)
        predict_v2(model, tok, val_w, device)
    log("done.")


def main_decode():
    # predict only, no training or corpus streaming. needs best.pt (ideally also the window cache) attached as input
    hf_login()
    os.makedirs(CKPT_DIR, exist_ok=True)
    device = setup_device()
    tok = load_tokenizer()
    model = build_model(tok, device)
    load_weights_for_prediction(model)
    model.eval()
    val_w = load_cached_val_windows()
    if RUN_SELF_TEST:
        self_test(model, val_w, device)
    predict_v2(model, tok, val_w, device)
    log("done.")


if __name__ == "__main__":
    main() if MODE == "train" else main_decode()
